环境：Apache Polaris (REST Catalog)-1.3.0 + Spark SQL-3.5.8 + Kyuubi-1.10.3 + MinIO

## 概念介绍
### 为什么需要隐藏分区
在传统的数仓或早期数据湖格式中，分区通常是显式的，这意味着分区列必须作为真实的物理列存储在每一行数据中，且目录结构直接暴露分区值。如果没有隐藏分区功能，数据团队将面临以下核心痛点：

+ **痛点一：存储浪费与数据冗余**
    - 为了支持按时间或类别查询，相同的分区值（如日期2026-03-14）必须在表中的每一行数据里重复存储。对于海量数据，这种无意义的重复会消耗巨大的存储空间，增加存储成本，同时降低压缩效率。
+ **痛点二：查询逻辑复杂且易错**
    - 用户必须确切知道底层的分区策略（是按天、按小时还是按字符串前缀），并在编写 SQL 时使用特定的函数或精确匹配条件才能触发分区裁剪。一旦查询条件与底层分区格式不完全匹配（例如底层按小时存，用户按天查），引擎就无法跳过无关文件，导致全表扫描，查询性能急剧下降。
+ **痛点三：架构演进极其困难**
    - 如果业务增长导致原有的分区粒度不再适用（例如从按天分区需要调整为按小时分区），在传统模式下，这通常意味着必须停止服务、重写所有历史数据文件并迁移整个目录结构。这种“牵一发而动全身”的耦合性使得数据架构难以适应快速变化的业务需求。

Iceberg 的解决方案： Iceberg 引入了隐藏分区机制，彻底解耦了物理存储布局与逻辑查询视图。

+ 核心价值：分区信息仅存在于元数据文件中，不作为普通列存储在数据文件内，对用户透明。
+ 自动变换：允许定义基于原始列的变换规则（如提取小时、截断字符串、哈希桶）。用户只需针对原始列编写简单的过滤条件（如 `WHERE timestamp > ...`），Iceberg 引擎会自动将其转换为对应的分区过滤逻辑，精准跳过无关文件。
+ 灵活演进：可以随时修改分区策略（如从按天改为按小时），而无需重写现有数据。新旧数据可以共存，引擎在查询时自动统一视图。

---

## 2. 核心概念详解
| 特性 | 传统显式分区(Hive) | Iceberg 隐藏分区 |
| --- | --- | --- |
| 数据存储 | 分区列值重复存储在每一行数据中 | 分区列值不存入数据文件，仅在元数据记录 |
| 查询写法 | 必须精确匹配分区列值或目录结构 | 直接过滤原始列，引擎自动推导分区范围 |
| 分区变换 | 不支持（存什么查什么） | 支持自动变换 (`hour`, `day`, `month`, `bucket`, `truncate`) |
| Schema 耦合 | 强耦合，改分区需重写数据 | 解耦，可随时修改分区策略而不重写数据 |
| 目录结构 | `/dt=2026-03-14/file.parquet` | `/data-uuid.parquet` |


---

## 实战案例：日志表的智能分区演进
场景：我们要建立一张用户行为日志表 `dataspire_catalog.db_dev.user_events`。

+ 需求：数据量巨大，主要按时间查询。
+ 策略：底层文件按 小时 (hour) 物理存储（适合流式写入，避免小文件过多），但希望用户能按 天 (day) 轻松查询，且不需要在表中看到冗余的日期列。

### 建表
注意 `PARTITIONED BY` 中使用的是 变换函数，而不是单纯的列名。这意味着 `event_time` 是原始列，而分区是基于它的变换结果。

```sql
-- 创建表
-- 策略：底层物理分区按小时切分，但对用户隐藏细节
CREATE TABLE dataspire_catalog.db_dev.user_events (
    event_id BIGINT,
    user_id STRING,
    event_type STRING,
    event_time TIMESTAMP,
    details STRING
) USING iceberg
PARTITIONED BY (
    hours(event_time)  -- 关键：定义隐藏分区变换规则
);
```

### 写入数据
写入时，不需要指定分区列的值，也不需要像 Hive 那样使用 `INSERT INTO ... PARTITION(...)` 语法。直接插入包含原始列的数据即可，Iceberg 会自动计算分区值并路由文件。

```sql
-- 模拟写入不同时间段的数据
INSERT INTO dataspire_catalog.db_dev.user_events VALUES
(1, 'u1', 'click', timestamp('2026-03-14 10:15:00'), 'page_A'),
(2, 'u2', 'view',  timestamp('2026-03-14 10:45:00'), 'page_B'),
(3, 'u3', 'buy',   timestamp('2026-03-14 11:20:00'), 'item_X'),
(4, 'u4', 'click', timestamp('2026-03-14 14:05:00'), 'page_C'),
(5, 'u5', 'view',  timestamp('2026-03-15 09:00:00'), 'home');

-- 验证：查看底层文件结构 (通过 system 表)
-- 你会发现 file_path 中没有 dt=xxx 的目录层级
SELECT file_path, partition 
FROM dataspire_catalog.db_dev.user_events.files 
LIMIT 5;
-- 输出示例: partition 列显示为内部编码 (如 {hours=15768})，而非直观字符串
```

### 查询演示
这是隐藏分区最强大的地方。用户只需对原始列`event_time` 进行过滤，无需知晓底层是按小时分区的。

#### 场景 A：按天查询
用户想查 2026-03-14 全天的数据。虽然底层分散在 10点、11点、14点等多个文件中，Iceberg 会自动识别并只读取这些文件。

```sql
-- 用户视角：简单的范围查询，无需任何分区函数
SELECT count(*) 
FROM dataspire_catalog.db_dev.user_events
WHERE event_time >= timestamp('2026-03-14 00:00:00')
  AND event_time <  timestamp('2026-03-15 00:00:00');

-- 原理：Spark 计划器将范围下推给 Iceberg，Iceberg 根据元数据
-- 计算出该范围覆盖了哪些 "hours" 分区，从而跳过其他文件。
```

#### 场景 B：按特定小时查询
```sql
SELECT * 
FROM dataspire_catalog.db_dev.user_events
WHERE event_time >= timestamp('2026-03-14 10:00:00')
  AND event_time <  timestamp('2026-03-14 11:00:00');
-- 结果：仅扫描 10:00-11:00 对应的物理文件，效率极高。
```

### 进阶：动态修改分区策略 
业务增长后，发现按小时分区导致元数据文件过多。现在想改成按 天 (day) 分区。 在传统模式下，这需要迁移 PB 级数据。在 Iceberg 中，只需一条 DDL 修改未来数据的分区方式。

```sql
-- 1. 修改分区策略：将未来的新数据改为按天分区
-- 旧数据保持按小时分区，新数据按天分区，两者共存
ALTER TABLE dataspire_catalog.db_dev.user_events
REPLACE PARTITION FIELD hours(event_time) WITH days(event_time);

-- 2. 写入新数据 (将按新的 day 策略存储)
INSERT INTO dataspire_catalog.db_dev.user_events VALUES
(6, 'u6', 'login', timestamp('2026-03-16 12:00:00'), 'app');

-- 3. 验证
-- 查询时，无论数据是旧的小时分区还是新的天分区，
-- 用户依然只需要写 WHERE event_time >= ...
-- Iceberg 会自动处理混合分区布局，对用户完全透明。
SELECT count(*) 
FROM dataspire_catalog.db_dev.user_events
WHERE event_time >= timestamp('2026-03-16 00:00:00');
```

关键点: `REPLACE PARTITION FIELD` 允许你在不移动任何现有数据文件的情况下，改变表的分区演进方向。

### 其他常用变换函数
除了时间变换，Iceberg 还支持多种隐藏变换策略：

```sql
-- 示例 1: 哈希分区 (解决数据倾斜，均匀分布文件)
CREATE TABLE dataspire_catalog.db_dev.user_profile_dist (
    user_id STRING,
    info STRING
) USING iceberg
PARTITIONED BY (
    bucket(16, user_id)  -- 将 user_id 哈希到 16 个桶中
);

-- 示例 2: 字符串截断分区 (按前缀分类，如国家代码)
CREATE TABLE dataspire_catalog.db_dev.global_logs (
    country_code STRING,
    msg STRING
) USING iceberg
PARTITIONED BY (
    truncate(2, country_code) -- 按 'US', 'CN', 'DE' 等前 2 位字符分区
);
```

---

## 最佳实践 
### 分区粒度选择
+ 流式写入: 推荐 `hours()`。避免 `minutes()`，会导致 Manifest 文件数量爆炸，影响元数据性能。
+ 批量写入: 推荐 `days()` 或 `months()`。
+ 判断标准: 目标是让每个分区文件的大小适中（通常建议 256MB - 1GB）。如果单个分区文件太小，说明分区过细；如果太大，说明分区过粗，不利于并发读取。

### 查询编写规范
+ 始终过滤原始列: 永远针对定义表时的原始列（如 `event_time`）编写 `WHERE` 子句。不要尝试去构造或过滤内部的分区值。
+ 利用范围查询: Iceberg 对范围查询 (`>`, `<`, `between`) 的分区裁剪支持最好。避免在过滤列上使用复杂函数（如 `date_format(event_time, ...)`），这可能导致引擎无法识别分区边界，从而退化为全表扫描。

### 演进策略
+ 先细后粗: 在业务初期不确定数据量时，可以使用较细的粒度（如 `hours`）。随着数据积累，如果发现小文件太多，可以使用 `ALTER TABLE ... REPLACE PARTITION FIELD` 切换到较粗粒度（如 `days`），而无需重写历史数据。
+ 监控元数据: 定期查看 `system.files` 或 `system.metadata_log`。如果 manifest 文件数量激增，说明分区可能过细，需要考虑合并或调整策略。

### 常见误区
+ 误区: “隐藏分区意味着没有物理目录结构，所以无法在 S3/MinIO 上直接通过文件夹浏览数据。”
    - 真相: 是的，这是设计意图。Iceberg 的数据文件命名是随机的 UUID，严禁直接通过文件系统路径来管理或查询数据。必须通过 Spark/Iceberg 引擎访问。
+ 误区: “修改分区策略后，旧数据会自动重组为新分区格式。”
    - 真相: 不会。旧数据保持原样，新数据按新策略写入。Iceberg 会在查询时自动合并视图。如果需要物理重组旧数据以统一文件格式，需手动运行 `rewrite_data_files`。

---

##  总结速查表
| 操作/特性 | 传统 Hive 分区 | Iceberg 隐藏分区 | 优势 |
| --- | --- | --- | --- |
| 建表语法 | `PARTITIONED BY (dt STRING)` | `PARTITIONED BY (days(ts))` | 支持自动变换 |
| 数据存储 | 每行都存 `dt`<br/> 值 | 仅元数据存，文件内无 `dt` | 节省存储，数据纯净 |
| 查询过滤 | `WHERE dt = '...'` | `WHERE ts >= '...'` | 语义直观，不易出错 |
| 修改分区 | 需重写全表数据 | `ALTER TABLE ... REPLACE ...` | 秒级切换，零停机 |
| 多粒度共存 | 不支持 | 支持 (新旧数据不同分区策略) | 灵活适应业务增长 |